In [2]:
import gymnasium as gym
import numpy as np
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
ENV_NAME = "MountainCar-v0"

def make_env(seed):
    env = gym.make(ENV_NAME)
    env.reset(seed=seed)

    # Override max episode steps to 2000 (IMPORTANT)
    env._max_episode_steps = 2000

    return env

In [11]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s_next, d = zip(*batch)

        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=device),
            torch.tensor(a, dtype=torch.long, device=device),
            torch.tensor(r, dtype=torch.float32, device=device),
            torch.tensor(np.array(s_next), dtype=torch.float32, device=device),
            torch.tensor(d, dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)

In [12]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

        self.apply(self.init_weights)

    def init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

In [13]:
GAMMA = 0.99
LR = 5e-4
BATCH_SIZE = 128
BUFFER_SIZE = 20000

EPS_START = 1.0
EPS_END = 0.05
EPS_DECAY = 100000   # slow decay important

TARGET_UPDATE = 2000
NUM_EPISODES = 1000
MAX_STEPS = 2000
replay_factor = rho = 1

In [14]:
def select_action(state, q_net, epsilon, action_dim):
    if random.random() < epsilon:
        return random.randrange(action_dim)

    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        return q_net(state).argmax(dim=1).item()

In [15]:
def train_dqn(seed=0):
    env = make_env(seed)

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    q_net = QNetwork(state_dim, action_dim).to(device)
    target_net = QNetwork(state_dim, action_dim).to(device)
    target_net.load_state_dict(q_net.state_dict())

    optimizer = optim.Adam(q_net.parameters(), lr=LR)
    replay_buffer = ReplayBuffer(BUFFER_SIZE)

    epsilon = EPS_START
    epsilon_decay = (EPS_START - EPS_END) / EPS_DECAY

    rewards = []
    total_steps = 0

    for episode in range(NUM_EPISODES):
        state, _ = env.reset()
        episode_reward = 0

        for step in range(MAX_STEPS):
            total_steps += 1

            action = select_action(state, q_net, epsilon, action_dim)
            next_state, reward, terminated, truncated, _ = env.step(action)

            done = terminated or truncated

            replay_buffer.push(state, action, reward, next_state, done)
            state = next_state
            episode_reward += reward

            # Learning step
            if len(replay_buffer) > BATCH_SIZE:
                s, a, r, s_next, d = replay_buffer.sample(BATCH_SIZE)

                q_values = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)

                with torch.no_grad():
                    max_next_q = target_net(s_next).max(1)[0]
                    target = r + GAMMA * max_next_q * (1 - d)

                loss = nn.MSELoss()(q_values, target)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # Hard target update
            if total_steps % TARGET_UPDATE == 0:
                target_net.load_state_dict(q_net.state_dict())

            # Epsilon decay
            epsilon = max(EPS_END, epsilon - epsilon_decay)

            if done:
                break

        rewards.append(episode_reward)

        if episode % 10 == 0:
            print(f"Episode {episode}, Reward: {episode_reward}, Epsilon: {epsilon:.3f}")

    env.close()
    return rewards

In [ ]:
all_rewards = []

for seed in range(15):
    rewards = train_dqn(seed)
    all_rewards.append(rewards)

**4 A**

In [ ]:
import re

def parse_rewards(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    all_runs = []
    current_run = []

    for line in lines:
        match = re.search(r"Episode (\d+), Reward: ([\-0-9.]+)", line)
        if match:
            episode = int(match.group(1))
            reward = float(match.group(2))

            if episode == 0 and len(current_run) > 0:
                all_runs.append(current_run)
                current_run = []

            current_run.append(reward)

    if current_run:
        all_runs.append(current_run)

    return all_runs

In [ ]:
data = {}
rhos = [1, 2, 4]

for rho in rhos:
    data[rho] = parse_rewards(f"/content/drive/MyDrive/rho{rho}.csv")

for rho in data:
    data[rho] = np.array(data[rho])

Episode 0, Reward: -2000.0, Epsilon: 0.981
Episode 10, Reward: -2000.0, Epsilon: 0.791
Episode 20, Reward: -2000.0, Epsilon: 0.601
Episode 30, Reward: -2000.0, Epsilon: 0.411
Episode 40, Reward: -2000.0, Epsilon: 0.221


KeyboardInterrupt: 

In [ ]:
def moving_average(x, window=20):
    return np.convolve(x, np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 6))

for rho in [1, 2, 4, 8]:
    all_rewards = np.array(data[rho])   

    mean_rewards = np.mean(all_rewards, axis=0)
    std = np.std(all_rewards, axis=0)
    n = all_rewards.shape[0]

    ci = 1.96 * std / np.sqrt(n)

    smooth_mean = moving_average(mean_rewards, window=20)
    smooth_ci = moving_average(ci, window=20)

    x = np.arange(len(smooth_mean))

    plt.plot(x, smooth_mean, label=f"ρ = {rho}", linewidth=2)
    plt.fill_between(
        x,
        smooth_mean - smooth_ci,
        smooth_mean + smooth_ci,
        alpha=0.2
    )

plt.xlabel("Episodes")
plt.ylabel("Total Reward")
plt.title("Training Performance (DQN on MountainCar, Smoothed)")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

def compute_auc_per_run(data_rho):
    # data_rho shape: (num_seeds, num_episodes)
    aucs = []

    for run in data_rho:
        auc = np.mean(run)   
        aucs.append(auc)

    return np.array(aucs)

In [ ]:
rhos = sorted(data.keys())

means = []
cis = []

for rho in rhos:
    runs = data[rho]  # shape: (seeds, episodes)

    aucs = [np.mean(run) for run in runs]

    mean_auc = np.mean(aucs)
    std = np.std(aucs)
    ci = 1.96 * std / np.sqrt(len(aucs))

    means.append(mean_auc)
    cis.append(ci)


plt.figure(figsize=(8,6))
x = np.arange(len(rhos))
plt.errorbar(
    x,
    means,
    yerr=cis,
    fmt='o',
    capsize=5
)

plt.plot(x, means, linestyle='-', color='gray', alpha=0.7)

plt.xticks(x, [f"ρ = {r}" for r in rhos])
plt.xlabel("Variant (Replay Factor ρ)")
plt.ylabel("Aggregate Performance")
plt.title("Comparison Plot 2 (95% Confidence Intervals)")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

4 part b

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

plt.figure(figsize=(10,6))

colors = {1: 'blue', 2: 'orange', 4: 'green', 8: 'red'}


all_auc = np.concatenate(list(auc_data.values()))
x_vals = np.linspace(-500, 0, 300)

# --- Plot ---
for rho in auc_data:
    perf = auc_data[rho]

    kde = gaussian_kde(perf)

    plt.plot(x_vals, kde(x_vals), color=colors[rho], label=f"ρ = {rho}")
    plt.fill_between(x_vals, kde(x_vals), color=colors[rho], alpha=0.2)

plt.xlabel("Aggregate Performance (AUC)")
plt.ylabel("Density")
plt.title("Performance Distribution Across Replay Factors")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

4 part c

In [3]:
from scipy.stats import norm, chi2

def compute_k(n, alpha=0.05, beta=0.9):
    z = norm.ppf((1 + beta) / 2)          # from beta
    chi = chi2.ppf(1 - alpha, df=n - 1)   # from alpha
    k = z * np.sqrt((n - 1) / chi)
    return k

def compute_tolerance_interval(data, alpha=0.05, beta=0.9):
    n = data.shape[0]
    k = compute_k(n, alpha, beta)

    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0)

    lower = mean - k * std
    upper = mean + k * std

    return mean, lower, upper, k

plt.figure(figsize=(10,6))

window = 20  # smoothing window

for rho in data:
    arr = np.array(data[rho])  
    mean, lower, upper, k = compute_tolerance_interval(arr)
    print(f"ρ = {rho}, k = {k:.3f}")  

    mean_s = moving_average(mean, window)
    lower_s = moving_average(lower, window)
    upper_s = moving_average(upper, window)

    x = np.arange(len(mean_s)) * 10  

    plt.plot(x, mean_s, label=f"ρ = {rho}", linewidth=2)
    plt.fill_between(x, lower_s, upper_s, alpha=0.2)

plt.xlabel("Episodes")
plt.ylabel("Return")
plt.title("Tolerance Intervals (α=0.05, β=0.9, Smoothed)")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined